In [1]:
#Python Modules
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import torch 
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"]   = "1"

#ML Modules
import train.train_code as tr_c
import preprocess.data_frame_preprocess as prep
import models.model_bge_m3 as bge_m3
import models.model_xgboost as xgb_m3
import evaluate.long_tokens
import evaluate.long_tokens as long_tok
import dataset.Train as multi_task

#Config FILE
import config as cfg

/home/andreoseo/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Transformer

In [2]:
#/home/andreoseo/Documents/MiSongGyny2026/DATA/train_w_gemma26b/resultado_train_drop_col.csv
cfg.train.LOSS_W1 = 1.0
cfg.train.LOSS_W2 = 0.0
cfg.train.LOSS_W3 = 0.0

cfg.paths.SAVE_DIR = '/home/andreoseo/Documents/MiSongGyny2026/models/checkpoints_true/bg3_gemma26b_t1'
#cfg.paths.CSV_PATH = '/home/andreoseo/Documents/MiSongGyny2026/DATA/train_w_gemma26b/resultado_train_drop_col.csv'

cfg.train.BEST_MODEL_METRIC  = 'f1_t1'
cfg.train.GRADIENT_CHECKPOINTING = False
cfg.train.BATCH_SIZE   = 4
cfg.train.EPOCHS = 15
cfg.train.SEED = 30

## Check Save Paths

In this part we review were we will save the weights and results

In [3]:
os.makedirs(cfg.paths.SAVE_DIR,exist_ok=True)
print(cfg.paths.SAVE_DIR)

/home/andreoseo/Documents/MiSongGyny2026/models/checkpoints_true/bg3_gemma26b_t1


## Prepare Data Frame

In [4]:
raw_df = pd.read_csv(cfg.paths.CSV_PATH)
df = prep.prepare_dataframe(raw_df,cfg) #New columns: lyrics_clean & gloss_clean

strat_key = (df['label_t1'].astype(str) + '_' +
             df['label_t3'].astype(str) + '_' +
             df[cfg.columns.LABEL_COLS_T2].astype(str).agg('-'.join, axis=1))

[METRICS] Palabras promedio PRE:  348.05
[METRICS] Palabras promedio POST: 295.09
[METRICS] Reducción de tokens:    15.22%
[FLAG] Usando Glosses para entrenamiento


## Load Models

In [5]:
#Load Transformer Model
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer,model = bge_m3.load_tokenizer_model(cfg,device)

Loading weights: 100%|█| 391/391 [00:00<00:00, 85629.33


In [ ]:
longitudes, estadisticas = long_tok.analizar_longitud_tokens(
    df['lyrics_clean'], tokenizer, cfg.tokenization.MAX_LEN,
    prefix=cfg.paths.MODEL_PREFIX,  # ahora puede ser ''
    verbose = True,
    add_verbose=df['type_violence']
)

print(df['type_violence'].sum())

In [7]:
print(df['type_violence'].sum())

314


## Create Datasets

In [6]:
# Generate DataSets and DataLoaders
train_df, val_df = train_test_split(df, test_size=cfg.train.TEST_SIZE,
                                    stratify=strat_key,
                                    random_state= cfg.train.SEED)

#Dataset
train_ds = multi_task.MultiTaskDataset(train_df, cfg,tokenizer, cfg.tokenization.MAX_LEN,
                            head_len=cfg.tokenization.HEAD_LEN,
                            model_prefix=cfg.paths.MODEL_PREFIX)
val_ds   = multi_task.MultiTaskDataset(val_df, cfg,  tokenizer, cfg.tokenization.MAX_LEN,
                            head_len=cfg.tokenization.HEAD_LEN,
                            model_prefix=cfg.paths.MODEL_PREFIX)
#DataLoader
train_loader = DataLoader(train_ds, batch_size=cfg.train.BATCH_SIZE, shuffle=True,
                          num_workers=6, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.train.BATCH_SIZE,
                          num_workers=6, pin_memory=True)

In [7]:
if cfg.tokenization.HEAD_LEN is False:
    print(f'  Truncación: solo cabeza (primeros {cfg.tokenization.MAX_LEN} tokens)')
else:
    print(f'  Truncación: Head+Tail  (head={cfg.tokenization.HEAD_LEN} | tail={cfg.tokenization.MAX_LEN - cfg.tokenization.HEAD_LEN - 2})')

  Truncación: solo cabeza (primeros 1024 tokens)


## Training - BGE M3

In [10]:
criterions = tr_c.build_criterions(
    train_df = train_df,
    t2_label_cols = cfg.columns.LABEL_COLS_T2,
    device = device
)

history, best_f1 = tr_c.train(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterions=criterions,
        tokenizer=tokenizer,
        cfg=cfg,
        device=device,
    )

[SETUP] Gradient checkpointing: OFF

[TRAINING] Iniciando entrenamiento multi-task…
[TRAINING] Métrica de guardado: f1_t1

  Epoch 1 | Step 20/403 | Loss: 0.6893 (T1=0.689 T2=1.477 T3=0.705)
  Epoch 1 | Step 40/403 | Loss: 0.6877 (T1=0.688 T2=1.075 T3=0.692)
  Epoch 1 | Step 60/403 | Loss: 0.6932 (T1=0.693 T2=1.028 T3=0.689)
  Epoch 1 | Step 80/403 | Loss: 0.6828 (T1=0.683 T2=1.067 T3=0.700)
  Epoch 1 | Step 100/403 | Loss: 0.6798 (T1=0.680 T2=1.107 T3=0.706)
  Epoch 1 | Step 120/403 | Loss: 0.7050 (T1=0.705 T2=0.746 T3=0.690)
  Epoch 1 | Step 140/403 | Loss: 0.6931 (T1=0.693 T2=1.307 T3=0.689)
  Epoch 1 | Step 160/403 | Loss: 0.6749 (T1=0.675 T2=0.794 T3=0.695)
  Epoch 1 | Step 180/403 | Loss: 0.6785 (T1=0.678 T2=0.702 T3=0.689)
  Epoch 1 | Step 200/403 | Loss: 0.6515 (T1=0.652 T2=0.702 T3=0.688)
  Epoch 1 | Step 220/403 | Loss: 0.6083 (T1=0.608 T2=0.786 T3=0.699)
  Epoch 1 | Step 240/403 | Loss: 0.5361 (T1=0.536 T2=0.783 T3=0.698)
  Epoch 1 | Step 260/403 | Loss: 0.6223 (T1=0.622 T2=

KeyboardInterrupt: 

# Train XGBOOST w Embeddings

In [8]:
# Cargar pesos entrenados
#tokenizer, model = load_tokenizer_model(cfg, device)
model.load_state_dict(
    torch.load(
        os.path.join(
            '/home/andreoseo/Documents/MiSongGyny2026/models/checkpoints_true/bg3_gemma26b_t1',
            'model_weights.pt'
        ),
        map_location=device
    )
)

<All keys matched successfully>

In [9]:
X_train, y_t1_train, y_t2_train, y_t3_train = xgb_m3.extract_embeddings_and_labels(model, train_loader, device)
X_val,   y_t1_val,   y_t2_val,   y_t3_val   = xgb_m3.extract_embeddings_and_labels(model, val_loader,   device)
#X_test_emb  = extract_embeddings(model, test_loader,  device)

In [12]:
import optuna
import xgboost as xgb
import numpy as np
from sklearn.metrics import f1_score, classification_report

optuna.logging.set_verbosity(optuna.logging.WARNING)

def train_binary_clf(X_train, y_train, X_val, y_val, n_trials=50, device='cuda'):
    """Entrena un clasificador binario XGBoost con Optuna para balanceo."""
    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()
    spw = neg / pos if pos > 0 else 1.0

    def objective(trial):
        clf = xgb.XGBClassifier(
            n_estimators     = trial.suggest_int  ('n_estimators',       100, 800),
            max_depth        = trial.suggest_int  ('max_depth',            3,   10),
            learning_rate    = trial.suggest_float('learning_rate',    0.005, 0.1, log=True),
            subsample        = trial.suggest_float('subsample',          0.5, 1.0),
            colsample_bytree = trial.suggest_float('colsample_bytree',   0.5, 0.9),
            min_child_weight = trial.suggest_int  ('min_child_weight',     1,  20),
            colsample_bylevel = trial.suggest_float('colsample_bylevel', 0.4, 1.0),
            colsample_bynode  = trial.suggest_float('colsample_bynode',  0.4, 1.0),
            reg_alpha         = trial.suggest_float('reg_alpha',    1e-4, 10.0, log=True),
            reg_lambda        = trial.suggest_float('reg_lambda',   1e-4, 10.0, log=True),
            gamma             = trial.suggest_float('gamma',           0.0,  5.0),
            #max_delta_step    = trial.suggest_int  ('max_delta_step',    1,   10),
            
            scale_pos_weight = spw,
            eval_metric      = 'logloss',
            device           = device,
            random_state     = 42,
            tree_method       = 'hist',
        )
        clf.fit(X_train, y_train)
        return f1_score(y_val, clf.predict(X_val))

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True, n_jobs=1)

    # reentrenar con mejores params
    best_clf = xgb.XGBClassifier(
        **study.best_params,
        scale_pos_weight = spw,
        eval_metric      = 'logloss',
        device           = device,
        random_state     = 42
    )
    best_clf.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
    early_stopping_rounds=30,  # evita overfitting en el refit final
)
    return best_clf, study.best_value, study.best_params


from sklearn.metrics import classification_report

# ==========================================
# --- Task 1 (Binaria) con Optuna ---
# ==========================================
print("\n" + "="*40)
print("── Task 1 (Binaria) | Optuna Optimization ──")
print("="*40)

# Llamamos a tu función ya definida
xgb_t1, t1_best_f1, t1_best_params = train_binary_clf(
    X_train=X_train, 
    y_train=y_t1_train, 
    X_val=X_val, 
    y_val=y_t1_val, 
    n_trials=200,       # Ajusta el número de trials según tu tiempo disponible
    device='cuda'      # Cambiar a 'cpu' si no usas GPU
)

print(f"\n  → Mejor F1-Score (Task 1): {t1_best_f1:.4f}")
print(f"  → Mejores Parámetros: {t1_best_params}")

print("\n── Classification Report T1 ──")
print(classification_report(y_t1_val, xgb_t1.predict(X_val)))




── Task 1 (Binaria) | Optuna Optimization ──


Best trial: 155. Best value: 0.830918: 100%|█| 200/200 


TypeError: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'

In [12]:
import joblib

In [13]:
joblib.dump(xgb_t1, '/home/andreoseo/Documents/MiSongGyny2026/models/checkpoints_true/xgboost_bg3_v3_t1_t3/xgb_t1.pkl')

['/home/andreoseo/Documents/MiSongGyny2026/models/checkpoints_true/xgboost_bg3_v3_t1_t3/xgb_t1.pkl']